# Detecção de Fraude em Transações com GNNs — Notebook Reprodutível (v4 • clean)

Este notebook consolida (em um único fluxo executável) os ajustes de rigor metodológico solicitados:

- **Setting temporal vs. transdutivo**: opção explícita e correção de *leakage* (estrutura e features).
- **Baselines não‑GNN**: **MLP (sem message passing)** + **XGBoost/LightGBM** (opcionais).
- **Capacidade comparável**: relatório de **parâmetros treináveis** e suporte a *equalização* por `hidden_dim`.
- **Early stopping alinhado**: monitoramento por **AP** e/ou **Recall@k%**.
- **Avaliação por limiar sem vazamento**: limiar **selecionado na validação** e aplicado no teste.
- **“Pureza” de casos/grupos**: substitui P@k(grupos) por métricas de **pureza/contaminação** consistentes.
- **Custo real**: tempo e memória por etapa (pré‑processamento, treino e inferência).

> Recomendação: execute as células em ordem. As seções “sweep” (profundidade/hyperparam/seed) são opcionais.


## Configurações de experimento

**Escolha o setting (importante para a banca):**

- `SETTING = "temporal"`: avaliação *realista* (sem usar arestas futuras na message passing; features agregadas só do histórico permitido).
- `SETTING = "transductive"`: assume grafo estático completo conhecido (permite usar a estrutura inteira na message passing). Ainda assim, **o scaler é ajustado no treino**.

**Observação sobre profundidade (100 camadas):** é possível testar, mas em GNNs “clássicas” isso costuma causar *oversmoothing/oversquashing* e piorar (ou explodir custo). No final há um *sweep* opcional com residual + LayerNorm para você testar de forma defensável.


In [ ]:
# %% [1] Setup: dependências, GPU/CPU e diretórios do projeto
import os, sys, re, subprocess
from pathlib import Path

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

# Dependências gerais
pip_install(["numpy","pandas","scikit-learn","matplotlib","networkx","psutil","tqdm"])

# PyTorch (normalmente já vem no Colab). Se não vier, instale aqui.
try:
    import torch
except Exception:
    # Ajuste o index-url se você estiver fora do Colab ou com outra CUDA.
    pip_install(["torch","torchvision","torchaudio","--index-url","https://download.pytorch.org/whl/cu121"])
    import torch

# PyG: instala wheels compatíveis com torch/cuda (evita compilar do source)
try:
    import torch_geometric  # noqa: F401
except Exception:
    torch_ver = torch.__version__.split("+")[0]
    cuda_ver  = torch.version.cuda  # ex: '12.1' ou None
    if cuda_ver is None:
        cu_tag = "cpu"
    else:
        m = re.match(r"(\d+)\.(\d+)", cuda_ver)
        cu_tag = f"cu{m.group(1)}{m.group(2)}" if m else "cpu"
    wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cu_tag}.html"
    print("➡️  Instalando PyG wheels em:", wheel_url)
    pip_install(["pyg_lib","torch_scatter","torch_sparse","torch_cluster","torch_spline_conv","-f", wheel_url])
    pip_install(["torch_geometric"])
    import torch_geometric  # noqa: F401

import numpy as np
import pandas as pd

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| device:", DEVICE)

# Tenta montar Drive se estiver no Colab (opcional)
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("[WARN] Não consegui montar o Drive:", repr(e))

# ====== CONFIG: ajuste o caminho do seu projeto ======
BASE = Path("/content/drive/MyDrive") if IN_COLAB else Path(".").resolve()

PROJECT_DIR = BASE / "DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k"  # <-- AJUSTE AQUI
DATA_DIR    = PROJECT_DIR

RESULTS_DIR = PROJECT_DIR / "results"
PLOTS_DIR   = PROJECT_DIR / "plots"
ARTIF_DIR   = PROJECT_DIR / "artifacts"

for d in [RESULTS_DIR, PLOTS_DIR, ARTIF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("PLOTS_DIR  :", PLOTS_DIR)
print("ARTIF_DIR  :", ARTIF_DIR)


In [ ]:
# %% [2] Configurações principais (setting, split, cache, execução)

# ====== SETTING (temporal vs transductivo) ======
SETTING = "temporal"     # "temporal" | "transductive"

# ====== Split temporal (frações) ======
TRAIN_FRAC = 0.60
VAL_FRAC   = 0.20

# ====== Cache do pré-processamento ======
CACHE_PATH   = ARTIF_DIR / "edge_data_v4_clean.pt"
FORCE_REBUILD = False

# ====== Debug (opcional): limitar arestas para testar rápido ======
DEBUG_MAX_EDGES = None   # ex: 2_000_000 para rodar rápido; None = usa tudo

# ====== Métricas de early stopping ======
MONITOR = "AP"           # "AP" | "R@2%" | "R@5%" ...
EVAL_EVERY = 5
PATIENCE   = 10
MAX_EPOCHS = 200

# ====== Avaliação: chunk de edges (reduz OOM em GPUs menores) ======
EVAL_CHUNK_SIZE = 100_000  # reduza para 50_000 se der OOM; aumente se tiver muita memória

# ====== PNA (pesado): habilite somente se tiver GPU/memória suficiente ======
INCLUDE_PNA = False        # True para incluir PNA na lista de GNNs
PNA_TOWERS = 2             # 1–2 recomendado p/ evitar OOM (4 pode estourar memória)
PNA_AGGR   = ["mean","min","max","std"]
PNA_SCALERS= ["identity","amplification","attenuation"]


# ====== Treino: balanceamento ======
BALANCE_TRAIN = True     # amostra negativos 1:1 com positivos no treino
LR = 1e-3
WD = 5e-5
DROPOUT = 0.2

# ====== Seeds ======
SEEDS = [42, 43, 44]     # aumente para mais robustez

# ====== Modelos a rodar ======
RUN_GNNS = True
RUN_MLP_BASELINE = True
RUN_XGB  = False   # opcional
RUN_LGBM = False   # opcional

print("SETTING:", SETTING)
print("CACHE :", CACHE_PATH)

In [ ]:
# %% [3] Leitura das transações + inferência robusta de colunas
from pathlib import Path

def norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    return df

def find_first_recursive(root: Path, candidates):
    for c in candidates:
        found = list(Path(root).rglob(c))
        if found:
            return found[0]
    return None

trx_path = find_first_recursive(DATA_DIR, [
    "transactions.csv","transaction.csv",
    "hi-large_trans.csv","hi-medium_trans.csv","hi-small_trans.csv",
    "li-large_trans.csv","li-medium_trans.csv","li-small_trans.csv",
])
assert trx_path is not None, f"Não encontrei CSV de transações em {DATA_DIR}"

transactions = norm_cols(pd.read_csv(trx_path))
print("Arquivo:", trx_path)
print("Colunas (amostra):", list(transactions.columns)[:20])
print("Linhas:", len(transactions))

def pick_col(df, candidates, name):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Não encontrei coluna {name}. Candidatas: {candidates}. Disponíveis: {list(df.columns)[:30]}...")

SRC_COL  = pick_col(transactions, ["sender_account_id","src","source","from","sender","src_id"], "SRC")
DST_COL  = pick_col(transactions, ["receiver_account_id","dst","dest","to","receiver","dst_id"], "DST")
AMT_COL  = pick_col(transactions, ["tx_amount","amount","amt","transaction_amount","value"], "AMOUNT")
TIME_COL = pick_col(transactions, ["timestamp","time","date","datetime","step"], "TIME")
TYPE_COL = pick_col(transactions, ["tx_type","type","transaction_type"], "TYPE")
Y_COL    = pick_col(transactions, ["is_fraud","is_laundering","fraud","label","y"], "LABEL")

print("Mapeamento de colunas:",
      "\n - SRC :", SRC_COL,
      "\n - DST :", DST_COL,
      "\n - AMT :", AMT_COL,
      "\n - TIME:", TIME_COL,
      "\n - TYPE:", TYPE_COL,
      "\n - Y   :", Y_COL)


In [ ]:
# %% [4] Construção do dataset de arestas (edge_index, edge_attr_raw, y, timestamp)
import numpy as np
import pandas as pd

df = transactions[[SRC_COL, DST_COL, AMT_COL, TIME_COL, TYPE_COL, Y_COL]].copy()

# Tipos
df[SRC_COL] = df[SRC_COL].astype(str)
df[DST_COL] = df[DST_COL].astype(str)
df[AMT_COL] = pd.to_numeric(df[AMT_COL], errors="coerce").fillna(0.0)
df[Y_COL]   = pd.to_numeric(df[Y_COL], errors="coerce").fillna(0).astype(int)

# Timestamp robusto -> int64 (ns) se datetime, ou int se numérico
if pd.api.types.is_numeric_dtype(df[TIME_COL]):
    ts = pd.to_numeric(df[TIME_COL], errors="coerce").fillna(0).astype(np.int64)
else:
    ts = pd.to_datetime(df[TIME_COL], errors="coerce")
    ts = ts.fillna(pd.Timestamp("1970-01-01"))
    ts = ts.astype("int64")  # ns

df["_ts"] = ts.values.astype(np.int64)

# Ordena por tempo (importante para split temporal e para delta_t)
df = df.sort_values("_ts").reset_index(drop=True)

# Delta t global (não vaza futuro; usa apenas ordem temporal)
dt = np.diff(df["_ts"].values, prepend=df["_ts"].values[0])
dt_sec = dt.astype(np.float64) / 1e9
df["_delta_t"] = dt_sec

# One-hot do tipo
type_dum = pd.get_dummies(df[TYPE_COL].astype(str), prefix="type", dtype=np.float32)

# Feature de amount (log1p para estabilidade numérica)
df["_log_amt"] = np.log1p(df[AMT_COL].values.astype(np.float64))

# Edge features RAW (sem scaler aqui)
ea_df_raw = pd.concat(
    [
        df[["_log_amt"]].astype(np.float32),
        type_dum.astype(np.float32),
        df[["_delta_t"]].astype(np.float32),
    ],
    axis=1
).reset_index(drop=True)

# Mapeia accounts -> node ids
all_ids = pd.Index(pd.concat([df[SRC_COL], df[DST_COL]], axis=0).unique())
id2idx  = pd.Series(np.arange(len(all_ids), dtype=np.int64), index=all_ids)

src_idx = id2idx.loc[df[SRC_COL]].values.astype(np.int64)
dst_idx = id2idx.loc[df[DST_COL]].values.astype(np.int64)

# Remove self-loops (opcional; mantém se quiser)
mask_valid = (src_idx != dst_idx)
if mask_valid.mean() < 1.0:
    print(f"[INFO] Removendo self-loops: {(~mask_valid).sum()} arestas")
src_idx = src_idx[mask_valid]
dst_idx = dst_idx[mask_valid]
ea_df_raw = ea_df_raw.iloc[mask_valid].reset_index(drop=True)
y_all = df.loc[mask_valid, Y_COL].values.astype(np.int64)
ts_all = df.loc[mask_valid, "_ts"].values.astype(np.int64)

# Opcional: limitar tamanho para debug
if DEBUG_MAX_EDGES is not None and len(y_all) > DEBUG_MAX_EDGES:
    print(f"[DEBUG] Subamostrando {DEBUG_MAX_EDGES} arestas (mantendo ordem temporal)")
    src_idx = src_idx[:DEBUG_MAX_EDGES]
    dst_idx = dst_idx[:DEBUG_MAX_EDGES]
    ea_df_raw = ea_df_raw.iloc[:DEBUG_MAX_EDGES].reset_index(drop=True)
    y_all = y_all[:DEBUG_MAX_EDGES]
    ts_all = ts_all[:DEBUG_MAX_EDGES]

edge_index_np = np.vstack([src_idx, dst_idx])  # (2, E)
E = edge_index_np.shape[1]
N = len(all_ids)

print(f"Nós (N): {N}")
print(f"Arestas (E): {E}")
print(f"Fraude (positivas): {int(y_all.sum())} ({100*y_all.mean():.6f}%)")
print("Dim edge_attr_raw:", ea_df_raw.shape)


In [ ]:
# %% [5] Split temporal (train/val/test) — sem vazamento e sem “ajuste no teste”
import numpy as np

def temporal_split(ts_int64, y, train_frac=0.6, val_frac=0.2):
    ts_int64 = np.asarray(ts_int64).astype(np.int64)
    y = np.asarray(y).astype(int)
    assert len(ts_int64) == len(y)
    thr1 = np.quantile(ts_int64, train_frac)
    thr2 = np.quantile(ts_int64, train_frac + val_frac)

    tr = ts_int64 <= thr1
    va = (ts_int64 > thr1) & (ts_int64 <= thr2)
    te = ts_int64 > thr2
    return tr, va, te, (thr1, thr2)

tr_mask, va_mask, te_mask, (thr1, thr2) = temporal_split(ts_all, y_all, TRAIN_FRAC, VAL_FRAC)

print("Split (frações reais):",
      "\n - train:", tr_mask.mean(),
      "\n - val  :", va_mask.mean(),
      "\n - test :", te_mask.mean())

print("Positivos por split:",
      "\n - train:", int(y_all[tr_mask].sum()),
      "\n - val  :", int(y_all[va_mask].sum()),
      "\n - test :", int(y_all[te_mask].sum()))

if y_all[tr_mask].sum() == 0 or y_all[va_mask].sum() == 0 or y_all[te_mask].sum() == 0:
    raise RuntimeError(
        "Split temporal ficou sem positivos em algum bloco. "
        "Ajuste TRAIN_FRAC/VAL_FRAC ou verifique o dataset."
    )

tr_idx = np.where(tr_mask)[0]
va_idx = np.where(va_mask)[0]
te_idx = np.where(te_mask)[0]

print("thr1, thr2 (timestamp):", thr1, thr2)


In [ ]:
# %% [6] (Re)construção sem leakage + cache (edge scaler train-only, node features, mp graphs)
import time
import torch
import inspect
import pickle
from sklearn.preprocessing import StandardScaler

def node_features_from_edges(edge_index_2xE, amount_raw, edge_mask, num_nodes):
    '''
    Features por nó (agregações simples) usando SOMENTE as arestas em edge_mask.
    Retorna (N, 6): [in_deg, out_deg, in_sum, out_sum, in_mean, out_mean]
    '''
    src = edge_index_2xE[0, edge_mask]
    dst = edge_index_2xE[1, edge_mask]
    amt = amount_raw[edge_mask].astype(np.float64)

    out_deg = np.bincount(src, minlength=num_nodes).astype(np.float64)
    in_deg  = np.bincount(dst, minlength=num_nodes).astype(np.float64)

    out_sum = np.bincount(src, weights=amt, minlength=num_nodes).astype(np.float64)
    in_sum  = np.bincount(dst, weights=amt, minlength=num_nodes).astype(np.float64)

    out_mean = out_sum / np.maximum(out_deg, 1.0)
    in_mean  = in_sum  / np.maximum(in_deg, 1.0)

    X = np.stack([in_deg, out_deg, in_sum, out_sum, in_mean, out_mean], axis=1).astype(np.float32)
    return X

def standardize_train_only(X, mask_rows):
    '''
    Ajusta mean/std APENAS nas linhas mask_rows e aplica em todas.
    '''
    X = np.asarray(X).astype(np.float32)
    mask_rows = np.asarray(mask_rows).astype(bool)
    mu = X[mask_rows].mean(axis=0)
    sd = X[mask_rows].std(axis=0)
    sd = np.where(sd < 1e-12, 1.0, sd).astype(np.float32)
    Xs = (X - mu) / sd
    return Xs.astype(np.float32), mu.astype(np.float32), sd.astype(np.float32)

def build_all_and_cache():
    t0 = time.perf_counter()

    # ----- Edge scaler (fit no treino; aplica em todos) -----
    ea_raw = ea_df_raw.values.astype(np.float32)
    scaler_e = StandardScaler()
    scaler_e.fit(ea_raw[tr_mask])
    ea_scaled = scaler_e.transform(ea_raw).astype(np.float32)

    # amount_raw para agregações (usa log_amt; evita amplitude extrema)
    amount_raw = ea_raw[:, 0].astype(np.float32)  # _log_amt

    # ----- Node features (sem leakage) -----
    # Treino: só arestas de treino
    X_tr_raw = node_features_from_edges(edge_index_np, amount_raw, tr_mask, N)
    # máscara de nós "vistos" no treino (para fit do scaler)
    seen_nodes = (X_tr_raw[:,0] + X_tr_raw[:,1]) > 0
    X_tr_scaled, mu_x, sd_x = standardize_train_only(X_tr_raw, seen_nodes)

    # Histórico maior (train+val) para inferência no teste (permitido no setting temporal)
    trainval_mask = tr_mask | va_mask
    X_trval_raw = node_features_from_edges(edge_index_np, amount_raw, trainval_mask, N)
    X_trval_scaled = ((X_trval_raw - mu_x) / sd_x).astype(np.float32)

    # Transductivo: usa todas as arestas para features, mas mantém scaler do treino
    X_all_raw = node_features_from_edges(edge_index_np, amount_raw, np.ones(E, dtype=bool), N)
    X_all_scaled = ((X_all_raw - mu_x) / sd_x).astype(np.float32)

    # ----- Message passing graphs -----
    if SETTING == "temporal":
        mp_train_mask = tr_mask
        mp_test_mask  = tr_mask | va_mask
        x_train_np = X_tr_scaled
        x_test_np  = X_trval_scaled
    elif SETTING == "transductive":
        mp_train_mask = np.ones(E, dtype=bool)
        mp_test_mask  = np.ones(E, dtype=bool)
        x_train_np = X_all_scaled
        x_test_np  = X_all_scaled
    else:
        raise ValueError("SETTING inválido: " + str(SETTING))

    # Tensors (mp em DEVICE; edge attrs/labels ficam em CPU para economizar VRAM)
    x_train = torch.tensor(x_train_np, dtype=torch.float32, device=DEVICE)
    x_test  = torch.tensor(x_test_np,  dtype=torch.float32, device=DEVICE)

    mp_ei_train = torch.tensor(edge_index_np[:, mp_train_mask], dtype=torch.long, device=DEVICE)
    mp_ei_test  = torch.tensor(edge_index_np[:, mp_test_mask],  dtype=torch.long, device=DEVICE)

    # Para GINE (edge-aware), precisamos de atributos nas arestas do mp grafo:
    mp_ea_train = torch.tensor(ea_scaled[mp_train_mask], dtype=torch.float32, device=DEVICE)
    mp_ea_test  = torch.tensor(ea_scaled[mp_test_mask],  dtype=torch.float32, device=DEVICE)

    # Arestas/attrs/labels completos em CPU (para decode em chunks)
    ei_all_cpu = torch.tensor(edge_index_np, dtype=torch.long, device="cpu")
    ea_all_cpu = torch.tensor(ea_scaled,     dtype=torch.float32, device="cpu")
    y_all_cpu  = torch.tensor(y_all,         dtype=torch.long, device="cpu")

    payload = {
        "SETTING": SETTING,
        "TRAIN_FRAC": TRAIN_FRAC,
        "VAL_FRAC": VAL_FRAC,
        "N": N,
        "E": E,
        "x_train": x_train,
        "x_test": x_test,
        "mp_ei_train": mp_ei_train,
        "mp_ei_test": mp_ei_test,
        "mp_ea_train": mp_ea_train,
        "mp_ea_test": mp_ea_test,
        "ei_all_cpu": ei_all_cpu,
        "ea_all_cpu": ea_all_cpu,
        "y_all_cpu": y_all_cpu,
        "tr_idx": tr_idx,
        "va_idx": va_idx,
        "te_idx": te_idx,
        "tr_mask": tr_mask,
        "va_mask": va_mask,
        "te_mask": te_mask,
        "scaler_e_mean": scaler_e.mean_.astype(np.float32),
        "scaler_e_scale": scaler_e.scale_.astype(np.float32),
        "mu_x": mu_x,
        "sd_x": sd_x,
    }

    dt = time.perf_counter() - t0
    payload["build_seconds"] = float(dt)
    return payload

# --- torch.load compat (PyTorch>=2.6 mudou default de weights_only para True) ---
# Este cache é gerado localmente por este notebook. Se você NÃO confia na origem do arquivo,
# defina FORCE_REBUILD=True ou apague o .pt para reconstruir ao invés de carregar via pickle.
def torch_load_cache(path, map_location="cpu"):
    try:
        sig = inspect.signature(torch.load)
        if "weights_only" in sig.parameters:
            # Para cache (dict com numpy arrays, etc.), precisamos do pickle completo.
            return torch.load(path, map_location=map_location, weights_only=False)
        return torch.load(path, map_location=map_location)
    except TypeError:
        # Versões antigas do torch sem argumento weights_only
        return torch.load(path, map_location=map_location)

# Cache
if CACHE_PATH.exists() and (not FORCE_REBUILD):
    print("[CACHE] Carregando:", CACHE_PATH)
    cache = torch_load_cache(CACHE_PATH, map_location="cpu")
    # Revalida config mínima
    if cache.get("SETTING") != SETTING or cache.get("TRAIN_FRAC") != TRAIN_FRAC or cache.get("VAL_FRAC") != VAL_FRAC:
        print("[CACHE] Config mudou (SETTING/split). Reconstruindo...")
        cache = build_all_and_cache()
        torch.save(cache, CACHE_PATH)
else:
    print("[CACHE] Construindo do zero...")
    cache = build_all_and_cache()
    torch.save(cache, CACHE_PATH)

# Extrai
x_train = cache["x_train"].to(DEVICE)
x_test  = cache["x_test"].to(DEVICE)

mp_ei_train = cache["mp_ei_train"].to(DEVICE)
mp_ei_test  = cache["mp_ei_test"].to(DEVICE)

mp_ea_train = cache["mp_ea_train"].to(DEVICE)
mp_ea_test  = cache["mp_ea_test"].to(DEVICE)

ei_all_cpu = cache["ei_all_cpu"]  # CPU
ea_all_cpu = cache["ea_all_cpu"]  # CPU
y_all_cpu  = cache["y_all_cpu"]   # CPU

tr_idx = cache["tr_idx"]; va_idx = cache["va_idx"]; te_idx = cache["te_idx"]

print("[OK] Cache pronto.",
      "\n - build_seconds:", cache.get("build_seconds"),
      "\n - x_train:", tuple(x_train.shape),
      "\n - edge_attr:", tuple(ea_all_cpu.shape),
      "\n - mp_ei_train edges:", mp_ei_train.shape[1],
      "\n - mp_ei_test  edges:", mp_ei_test.shape[1])


In [ ]:
# %% [7] Modelos: GNN encoders + Edge decoder + MLP baseline (sem message passing)
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv, GATv2Conv, GINEConv, PNAConv
from torch_geometric.utils import degree

IN_NODE = x_train.shape[1]
IN_EDGE = ea_all_cpu.shape[1]

class EdgeHead(nn.Module):
    # Decoder de arestas: concat(h_u, h_v, edge_attr) -> logit
    def __init__(self, h: int, in_edge: int, p: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2*h + in_edge, h),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(h, 1),
        )

    def forward(self, h_nodes, edge_index_2xB, edge_attr_BxF):
        src = edge_index_2xB[0]
        dst = edge_index_2xB[1]
        z = torch.cat([h_nodes[src], h_nodes[dst], edge_attr_BxF], dim=1)
        return self.net(z).squeeze(-1)

class SAGEEncoder(nn.Module):
    def __init__(self, in_dim: int, h: int, layers: int = 2, p: float = 0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.p = p
        if layers < 1:
            raise ValueError("layers precisa ser >= 1")
        self.convs.append(SAGEConv(in_dim, h))
        self.norms.append(nn.LayerNorm(h))
        for _ in range(layers - 1):
            self.convs.append(SAGEConv(h, h))
            self.norms.append(nn.LayerNorm(h))

    def forward(self, x, edge_index, edge_attr=None):
        h = x
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h_new = norm(h_new)
            h_new = F.relu(h_new)
            h_new = F.dropout(h_new, p=self.p, training=self.training)
            if h_new.shape == h.shape:
                h = h + h_new
            else:
                h = h_new
        return h

class GATv2Encoder(nn.Module):
    def __init__(self, in_dim: int, h: int, layers: int = 2, heads: int = 2, p: float = 0.2):
        super().__init__()
        self.p = p
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        self.convs.append(GATv2Conv(in_dim, h // heads, heads=heads, dropout=p))
        self.norms.append(nn.LayerNorm(h))

        for _ in range(layers - 1):
            self.convs.append(GATv2Conv(h, h // heads, heads=heads, dropout=p))
            self.norms.append(nn.LayerNorm(h))

    def forward(self, x, edge_index, edge_attr=None):
        h = x
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h_new = norm(h_new)
            h_new = F.elu(h_new)
            h_new = F.dropout(h_new, p=self.p, training=self.training)
            if h_new.shape == h.shape:
                h = h + h_new
            else:
                h = h_new
        return h

class GINEEncoder(nn.Module):
    # Encoder edge-aware: usa edge_attr nas mensagens
    def __init__(self, in_dim: int, in_edge: int, h: int, layers: int = 2, p: float = 0.2):
        super().__init__()
        self.p = p
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        def mlp(in_d, out_d):
            return nn.Sequential(nn.Linear(in_d, out_d), nn.ReLU(), nn.Linear(out_d, out_d))

        self.convs.append(GINEConv(mlp(in_dim, h), edge_dim=in_edge))
        self.norms.append(nn.LayerNorm(h))
        for _ in range(layers - 1):
            self.convs.append(GINEConv(mlp(h, h), edge_dim=in_edge))
            self.norms.append(nn.LayerNorm(h))

    def forward(self, x, edge_index, edge_attr):
        h = x
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index, edge_attr)
            h_new = norm(h_new)
            h_new = F.relu(h_new)
            h_new = F.dropout(h_new, p=self.p, training=self.training)
            if h_new.shape == h.shape:
                h = h + h_new
            else:
                h = h_new
        return h

def compute_deg_hist(edge_index_2xE, num_nodes: int):
    deg = degree(edge_index_2xE[0].cpu(), num_nodes=num_nodes).to(torch.long)
    hist = torch.bincount(deg, minlength=int(deg.max().item()) + 1).float()
    return hist

class PNAEncoder(nn.Module):
    def __init__(self, in_dim: int, h: int, deg_hist: torch.Tensor, layers: int = 2, towers: int = 2, aggr=None, scalers=None, p: float = 0.2):
        super().__init__()
        self.p = p
        if aggr is None:
            aggr = ["mean","min","max","std"]
        if scalers is None:
            scalers = ["identity","amplification","attenuation"]
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.convs.append(PNAConv(in_dim, h, aggr, scalers, deg=deg_hist, towers=towers, edge_dim=None))
        self.norms.append(nn.LayerNorm(h))
        for _ in range(layers - 1):
            self.convs.append(PNAConv(h, h, aggr, scalers, deg=deg_hist, towers=towers, edge_dim=None))
            self.norms.append(nn.LayerNorm(h))

    def forward(self, x, edge_index, edge_attr=None):
        h = x
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h_new = norm(h_new)
            h_new = F.relu(h_new)
            h_new = F.dropout(h_new, p=self.p, training=self.training)
            if h_new.shape == h.shape:
                h = h + h_new
            else:
                h = h_new
        return h

class GNNEdgeModel(nn.Module):
    def __init__(self, encoder: nn.Module, h_dim: int, in_edge: int, p: float = 0.2):
        super().__init__()
        self.encoder = encoder
        self.head = EdgeHead(h_dim, in_edge, p=p)

    def encode(self, x, mp_ei, mp_ea=None):
        try:
            return self.encoder(x, mp_ei, mp_ea)
        except TypeError:
            return self.encoder(x, mp_ei)

    def decode(self, h_nodes, eidx_2xB, eattr_BxF):
        return self.head(h_nodes, eidx_2xB, eattr_BxF)

# Baseline tabular: MLP sem message passing
class MLPBaseline(nn.Module):
    def __init__(self, in_dim: int, h: int = 256, p: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(h, h),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(h, 1),
        )

    def forward(self, X):
        return self.net(X).squeeze(-1)

def count_trainable_params(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

print("IN_NODE:", IN_NODE, "| IN_EDGE:", IN_EDGE)


In [ ]:
# %% [8] Métricas: AP/AUC, Recall@k, Lift@k, limiar em validação, pureza de grupos e custo
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, precision_recall_fscore_support

def topk_metrics(prob, y_true, ks=(0.01,0.02,0.05,0.10)):
    prob = np.asarray(prob).astype(float)
    y_true = np.asarray(y_true).astype(int)
    order = np.argsort(-prob)
    E = len(prob)
    base_prev = float(y_true.mean()) if E > 0 else np.nan
    out = {}
    for k in ks:
        m = max(1, int(round(E * k)))
        sel = order[:m]
        pos = int(y_true[sel].sum())
        prec = pos / max(m, 1)
        rec  = pos / max(int(y_true.sum()), 1)
        lift = (prec / base_prev) if base_prev > 0 else np.nan
        out[f"R@{int(k*100)}%"] = float(rec)
        out[f"L@{int(k*100)}%"] = float(lift)
        out[f"P@{int(k*100)}%"] = float(prec)
    return out

def basic_metrics(prob, y_true):
    prob = np.asarray(prob).astype(float)
    y_true = np.asarray(y_true).astype(int)
    out = {}
    try:
        out["AUC"] = float(roc_auc_score(y_true, prob))
    except Exception:
        out["AUC"] = np.nan
    try:
        out["AP"] = float(average_precision_score(y_true, prob))
    except Exception:
        out["AP"] = np.nan
    out.update(topk_metrics(prob, y_true))
    out["pos_rate"] = float(y_true.mean()) if len(y_true) else np.nan
    out["n_edges"] = int(len(y_true))
    return out

# Avaliação por limiar (sem vazamento): limiar escolhido na validação
def select_threshold_max_f1(y_val, p_val):
    y_val = np.asarray(y_val).astype(int)
    p_val = np.asarray(p_val).astype(float)
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    prec2 = prec[:-1]; rec2 = rec[:-1]; thr2 = thr
    f1 = 2 * prec2 * rec2 / np.maximum(prec2 + rec2, 1e-12)
    i = int(np.nanargmax(f1)) if len(f1) else 0
    return float(thr2[i]), float(prec2[i]), float(rec2[i]), float(f1[i])

def apply_threshold(y, p, thr):
    y = np.asarray(y).astype(int)
    p = np.asarray(p).astype(float)
    yhat = (p >= thr).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
    neg = (y == 0)
    far = float(((yhat == 1) & neg).sum() / max(int(neg.sum()), 1))
    return {"thr": float(thr), "Prec": float(prec), "Recall": float(rec), "F1": float(f1), "FAR": float(far)}

# “Pureza” de grupos (vetorizado)
import networkx as nx

def group_purity_stats(y_edge, edge_index_2xE, scores, k_pct=1.0, min_nodes=3):
    '''
    1) Seleciona top-k% arestas por score.
    2) Componentes (em nós) via arestas selecionadas (grafo não-direcionado).
    3) Pureza por componente olhando TODAS as arestas internas ao componente:
         purity = (#arestas positivas internas) / (#arestas internas)

    Observação: também retorna pureza usando apenas as arestas selecionadas (útil para “casos” investigados).
    '''
    y_edge = np.asarray(y_edge).astype(int)
    scores = np.asarray(scores).astype(float)
    E = len(y_edge)
    k = max(1, int(np.ceil((k_pct/100.0) * E)))
    order = np.argsort(-scores)
    sel = order[:k]

    u = edge_index_2xE[0].astype(np.int64)
    v = edge_index_2xE[1].astype(np.int64)

    u_sel = u[sel]
    v_sel = v[sel]

    # Componentes geradas pelas arestas selecionadas
    Gsel = nx.Graph()
    Gsel.add_edges_from(zip(u_sel.tolist(), v_sel.tolist()))
    comps = [c for c in nx.connected_components(Gsel) if len(c) >= min_nodes]
    n_groups = len(comps)

    if n_groups == 0:
        return {
            "k%": float(k_pct),
            "n_groups": 0,
            "purity_macro_selected": np.nan,
            "purity_weighted_selected": np.nan,
            "purity_macro_induced": np.nan,
            "purity_weighted_induced": np.nan,
            "edges_in_groups_induced": 0,
            "pos_in_groups_induced": 0,
        }

    # nó -> gid (array)
    num_nodes = int(edge_index_2xE.max()) + 1
    gid_of_node = np.full(num_nodes, -1, dtype=np.int64)
    for gid, nodes in enumerate(comps):
        gid_of_node[np.fromiter((int(n) for n in nodes), dtype=np.int64)] = gid

    # ===== Pureza usando somente arestas selecionadas =====
    y_sel = y_edge[sel].astype(np.int64)
    g_sel = gid_of_node[u_sel]  # cada aresta selecionada pertence ao componente de u (== v)

    # Observação: nós/arestas em componentes menores que `min_nodes` não entram em `comps`,
    # portanto ficam com gid = -1. Filtramos para evitar erro no np.bincount e manter consistência.
    valid_sel = (g_sel >= 0)
    g_sel_v = g_sel[valid_sel]
    y_sel_v = y_sel[valid_sel]

    edge_counts_sel = np.bincount(g_sel_v, minlength=n_groups).astype(np.int64) if g_sel_v.size else np.zeros(n_groups, dtype=np.int64)
    pos_counts_sel  = np.bincount(g_sel_v, weights=y_sel_v, minlength=n_groups).astype(np.float64) if g_sel_v.size else np.zeros(n_groups, dtype=np.float64)
    pur_sel = pos_counts_sel / np.maximum(edge_counts_sel, 1)
    purity_macro_sel = float(np.mean(pur_sel))
    purity_weighted_sel = float(pos_counts_sel.sum() / max(int(edge_counts_sel.sum()), 1))

    # ===== Pureza induzida olhando TODAS as arestas internas =====
    gu = gid_of_node[u]
    gv = gid_of_node[v]
    mask_in = (gu >= 0) & (gu == gv)
    gu_in = gu[mask_in]
    y_in  = y_edge[mask_in].astype(np.int64)

    edge_counts = np.bincount(gu_in, minlength=n_groups).astype(np.int64)
    pos_counts  = np.bincount(gu_in, weights=y_in, minlength=n_groups).astype(np.float64)

    pur = pos_counts / np.maximum(edge_counts, 1)
    purity_macro = float(np.mean(pur))
    purity_weighted = float(pos_counts.sum() / max(int(edge_counts.sum()), 1))

    return {
        "k%": float(k_pct),
        "n_groups": int(n_groups),
        "purity_macro_selected": purity_macro_sel,
        "purity_weighted_selected": purity_weighted_sel,
        "purity_macro_induced": purity_macro,
        "purity_weighted_induced": purity_weighted,
        "edges_in_groups_induced": int(edge_counts.sum()),
        "pos_in_groups_induced": int(pos_counts.sum()),
    }

# Custo (tempo/memória)
import os, psutil

def mem_rss_gb():
    proc = psutil.Process(os.getpid())
    return float(proc.memory_info().rss / (1024**3))

def gpu_peak_gb():
    try:
        import torch
        if torch.cuda.is_available():
            return float(torch.cuda.max_memory_allocated() / (1024**3))
    except Exception:
        pass
    return np.nan


In [ ]:
# %% [9] Treino/eval: GNNs e baselines (early stopping em AP/Recall@k; limiar escolhido na validação)
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import gc


def cleanup_cuda():
    """Libera cache CUDA e força coleta de lixo. Útil entre execuções para evitar OOM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def set_seed_all(seed: int):
    import random, os
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def predict_proba_edges(model: GNNEdgeModel,
                        x: torch.Tensor,
                        mp_ei: torch.Tensor,
                        mp_ea: torch.Tensor,
                        idx_edges_np: np.ndarray,
                        chunk_size: int = 250_000):
    model.eval()
    cleanup_cuda()

    h = model.encode(x, mp_ei, mp_ea)

    probs = []
    for s in range(0, len(idx_edges_np), chunk_size):
        idx = idx_edges_np[s:s+chunk_size]
        eidx = ei_all_cpu[:, idx].to(DEVICE, non_blocking=True)
        eattr = ea_all_cpu[idx].to(DEVICE, non_blocking=True)
        logits = model.decode(h, eidx, eattr)
        probs.append(torch.sigmoid(logits).detach().cpu())
    return torch.cat(probs, dim=0).numpy()

def train_gnn(model_name: str,
              model_ctor,
              seed: int,
              monitor: str = "AP",
              max_epochs: int = 200,
              patience: int = 10,
              eval_every: int = 5,
              lr: float = 1e-3,
              wd: float = 5e-5,
              balance_train: bool = True,
              chunk_size_eval: int = 250_000):

    set_seed_all(seed)
    cleanup_cuda()
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None

    model = model_ctor().to(DEVICE)
    params = count_trainable_params(model)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

    # índices de treino (CPU)
    y_tr = y_all_cpu.numpy()[tr_idx].astype(int)
    pos_tr = tr_idx[y_tr == 1]
    neg_tr = tr_idx[y_tr == 0]

    best_score = -1e18
    best_state = None
    best_epoch = None
    bad = 0

    t_train0 = time.perf_counter()
    rss0 = mem_rss_gb()

    for ep in range(1, max_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        # message passing embeddings (full-batch)
        h_nodes = model.encode(x_train, mp_ei_train, mp_ea_train)

        # batch de arestas para loss (balanceado ou não)
        if balance_train and len(pos_tr) > 0:
            rng = np.random.default_rng(seed + ep)
            take_neg = min(len(neg_tr), len(pos_tr))
            neg_s = rng.choice(neg_tr, size=take_neg, replace=False) if take_neg > 0 else np.array([], dtype=int)
            batch_idx = np.concatenate([pos_tr, neg_s]).astype(np.int64)
        else:
            batch_idx = tr_idx

        eidx = ei_all_cpu[:, batch_idx].to(DEVICE, non_blocking=True)
        eattr = ea_all_cpu[batch_idx].to(DEVICE, non_blocking=True)
        yb = y_all_cpu[batch_idx].to(DEVICE, non_blocking=True).float()

        logits = model.decode(h_nodes, eidx, eattr)
        loss = F.binary_cross_entropy_with_logits(logits, yb)
        loss.backward()
        opt.step()

        if (ep % eval_every) == 0:
            p_val = predict_proba_edges(model, x_train, mp_ei_train, mp_ea_train, va_idx, chunk_size=chunk_size_eval)
            y_val = y_all_cpu.numpy()[va_idx].astype(int)
            m_val = basic_metrics(p_val, y_val)

            if monitor == "AP":
                score = m_val["AP"]
            elif monitor.startswith("R@"):
                score = m_val.get(monitor, np.nan)
            else:
                raise ValueError("Monitor inválido: " + str(monitor))

            score_num = -1e18 if (score is None or np.isnan(score)) else float(score)

            if monitor == "AP":
                print(f"[{model_name} | seed={seed}] ep={ep:03d} loss={loss.item():.4f}  val_AP={score_num:.4f}")
            else:
                print(f"[{model_name} | seed={seed}] ep={ep:03d} loss={loss.item():.4f}  "
                      f"val_AP={m_val.get('AP',np.nan):.4f}  val_{monitor}={score_num:.4f}")

            if score_num > best_score:
                best_score = score_num
                best_epoch = ep
                best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    print(f"[{model_name}] Early stopping (sem melhora em {patience} avaliações).")
                    break

    train_seconds = float(time.perf_counter() - t_train0)
    rss1 = mem_rss_gb()
    gpu_peak = gpu_peak_gb()

    if best_state is not None:
        model.load_state_dict(best_state)

    # Probabilidades finais em val e test
    p_val = predict_proba_edges(model, x_train, mp_ei_train, mp_ea_train, va_idx, chunk_size=chunk_size_eval)
    y_val = y_all_cpu.numpy()[va_idx].astype(int)

    p_test = predict_proba_edges(model, x_test, mp_ei_test, mp_ea_test, te_idx, chunk_size=chunk_size_eval)
    y_test = y_all_cpu.numpy()[te_idx].astype(int)

    m_val = basic_metrics(p_val, y_val)
    m_test = basic_metrics(p_test, y_test)

    thr, _, _, _ = select_threshold_max_f1(y_val, p_val)
    m_thr_test = apply_threshold(y_test, p_test, thr)

    # Pureza de grupos no teste
    ei_test_np = ei_all_cpu.numpy()[:, te_idx]
    rows_purity = []
    for k_pct in [1,2,5,10]:
        rows_purity.append(group_purity_stats(y_test, ei_test_np, p_test, k_pct=k_pct))
    purity_df = pd.DataFrame(rows_purity)

    # Salva probs para reuso
    PROBS_DIR = RESULTS_DIR / "probs_v4"
    PROBS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(PROBS_DIR / f"{model_name}_seed{seed}_val.npz",  y=y_val,  p=p_val)
    np.savez_compressed(PROBS_DIR / f"{model_name}_seed{seed}_test.npz", y=y_test, p=p_test)

    out = {
        "model": model_name,
        "seed": int(seed),
        "setting": SETTING,
        "params": int(params),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "best_val_score": float(best_score),
        "train_seconds": train_seconds,
        "rss_gb_start": float(rss0),
        "rss_gb_end": float(rss1),
        "gpu_peak_gb": float(gpu_peak),
        "thr_val_maxf1": float(thr),
    }
    out.update({f"val_{k}": v for k,v in m_val.items()})
    out.update({f"test_{k}": v for k,v in m_test.items()})
    out.update({f"test_thr_{k}": v for k,v in m_thr_test.items() if k != "thr"})
    out["test_purity_macro_selected@1%"] = float(purity_df.loc[purity_df["k%"]==1, "purity_macro_selected"].values[0])
    out["test_purity_weighted_selected@1%"] = float(purity_df.loc[purity_df["k%"]==1, "purity_weighted_selected"].values[0])
    out["test_purity_macro_induced@1%"] = float(purity_df.loc[purity_df["k%"]==1, "purity_macro_induced"].values[0])
    out["test_purity_weighted_induced@1%"] = float(purity_df.loc[purity_df["k%"]==1, "purity_weighted_induced"].values[0])
    return out, purity_df

# ===== Baseline: MLP sem message passing (usa features por aresta) =====
def edge_features_torch(x_nodes: torch.Tensor, idx_edges_np: np.ndarray):
    idx = idx_edges_np
    src = ei_all_cpu.numpy()[0, idx]
    dst = ei_all_cpu.numpy()[1, idx]
    x_src = x_nodes[torch.tensor(src, device=DEVICE)]
    x_dst = x_nodes[torch.tensor(dst, device=DEVICE)]
    eattr = ea_all_cpu[idx].to(DEVICE)
    return torch.cat([x_src, x_dst, eattr], dim=1)

@torch.no_grad()
def predict_mlp(mlp: MLPBaseline, x_nodes: torch.Tensor, idx_edges_np: np.ndarray, chunk_size: int = 250_000):
    mlp.eval()
    probs=[]
    for s in range(0, len(idx_edges_np), chunk_size):
        idx = idx_edges_np[s:s+chunk_size]
        Xb = edge_features_torch(x_nodes, idx)
        logits = mlp(Xb)
        probs.append(torch.sigmoid(logits).detach().cpu())
    return torch.cat(probs, dim=0).numpy()

def train_mlp_baseline(seed: int,
                       h: int = 256,
                       p: float = 0.2,
                       lr: float = 1e-3,
                       wd: float = 0.0,
                       max_epochs: int = 50,
                       patience: int = 5,
                       eval_every: int = 2,
                       balance_train: bool = True,
                       chunk_size_eval: int = 100_000):
    set_seed_all(seed)
    cleanup_cuda()
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None

    in_dim = 2 * IN_NODE + IN_EDGE
    mlp = MLPBaseline(in_dim, h=h, p=p).to(DEVICE)
    params = count_trainable_params(mlp)
    opt = torch.optim.Adam(mlp.parameters(), lr=lr, weight_decay=wd)

    y_tr = y_all_cpu.numpy()[tr_idx].astype(int)
    pos_tr = tr_idx[y_tr == 1]
    neg_tr = tr_idx[y_tr == 0]

    best = -1e18
    best_state = None
    best_epoch = None
    bad = 0

    t0 = time.perf_counter()
    rss0 = mem_rss_gb()

    for ep in range(1, max_epochs+1):
        mlp.train()
        opt.zero_grad(set_to_none=True)

        if balance_train and len(pos_tr) > 0:
            rng = np.random.default_rng(seed + ep)
            take_neg = min(len(neg_tr), len(pos_tr))
            neg_s = rng.choice(neg_tr, size=take_neg, replace=False) if take_neg>0 else np.array([], dtype=int)
            batch_idx = np.concatenate([pos_tr, neg_s]).astype(np.int64)
        else:
            batch_idx = tr_idx

        Xb = edge_features_torch(x_train, batch_idx)
        yb = y_all_cpu[batch_idx].to(DEVICE).float()

        logits = mlp(Xb)
        loss = F.binary_cross_entropy_with_logits(logits, yb)
        loss.backward()
        opt.step()

        if ep % eval_every == 0:
            p_val = predict_mlp(mlp, x_train, va_idx, chunk_size=chunk_size_eval)
            y_val = y_all_cpu.numpy()[va_idx].astype(int)
            m_val = basic_metrics(p_val, y_val)
            score = m_val["AP"]
            print(f"[MLP | seed={seed}] ep={ep:03d} loss={loss.item():.4f} val_AP={score:.4f}")
            if score > best:
                best = score
                best_epoch = ep
                best_state = {k: v.detach().cpu().clone() for k,v in mlp.state_dict().items()}
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    break

    train_seconds = float(time.perf_counter() - t0)
    rss1 = mem_rss_gb()
    gpu_peak = gpu_peak_gb()

    if best_state is not None:
        mlp.load_state_dict(best_state)

    p_val = predict_mlp(mlp, x_train, va_idx, chunk_size=chunk_size_eval)
    y_val = y_all_cpu.numpy()[va_idx].astype(int)
    p_test = predict_mlp(mlp, x_test, te_idx, chunk_size=chunk_size_eval)
    y_test = y_all_cpu.numpy()[te_idx].astype(int)

    m_val = basic_metrics(p_val, y_val)
    m_test = basic_metrics(p_test, y_test)

    thr, _, _, _ = select_threshold_max_f1(y_val, p_val)
    m_thr_test = apply_threshold(y_test, p_test, thr)

    PROBS_DIR = RESULTS_DIR / "probs_v4"
    PROBS_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(PROBS_DIR / f"MLP_seed{seed}_val.npz",  y=y_val,  p=p_val)
    np.savez_compressed(PROBS_DIR / f"MLP_seed{seed}_test.npz", y=y_test, p=p_test)

    out = {
        "model": "MLP",
        "seed": int(seed),
        "setting": SETTING,
        "params": int(params),
        "best_epoch": int(best_epoch) if best_epoch is not None else None,
        "best_val_score": float(best),
        "train_seconds": train_seconds,
        "rss_gb_start": float(rss0),
        "rss_gb_end": float(rss1),
        "gpu_peak_gb": float(gpu_peak),
        "thr_val_maxf1": float(thr),
    }
    out.update({f"val_{k}": v for k,v in m_val.items()})
    out.update({f"test_{k}": v for k,v in m_test.items()})
    out.update({f"test_thr_{k}": v for k,v in m_thr_test.items() if k != "thr"})
    return out


In [ ]:
# %% [10] (Opcional) XGBoost / LightGBM (tabular) — sem message passing
# Atenção: para datasets muito grandes, use subamostragem de negativos (NEG_RATIO) e/ou treino incremental.

RUN_XGB  = RUN_XGB
RUN_LGBM = RUN_LGBM

NEG_RATIO = 5   # usa até NEG_RATIO negativos por positivo no treino (evita explodir RAM)

if RUN_XGB or RUN_LGBM:
    from sklearn.model_selection import train_test_split

def build_tabular_np(x_nodes_np, idx_edges_np):
    idx = idx_edges_np.astype(np.int64)
    src = edge_index_np[0, idx]
    dst = edge_index_np[1, idx]
    X = np.concatenate([x_nodes_np[src], x_nodes_np[dst], ea_all_cpu.numpy()[idx]], axis=1).astype(np.float32)
    return X

x_train_np = x_train.detach().cpu().numpy().astype(np.float32)
x_test_np  = x_test.detach().cpu().numpy().astype(np.float32)

y_tr_full = y_all[tr_idx].astype(int)
pos_tr = tr_idx[y_tr_full == 1]
neg_tr = tr_idx[y_tr_full == 0]

rng = np.random.default_rng(SEEDS[0])

if NEG_RATIO is None:
    tr_used = tr_idx
else:
    n_pos = len(pos_tr)
    n_neg_keep = min(len(neg_tr), int(max(n_pos, 1) * float(NEG_RATIO)))
    neg_keep = rng.choice(neg_tr, size=n_neg_keep, replace=False) if n_neg_keep>0 else np.array([], dtype=int)
    tr_used = np.concatenate([pos_tr, neg_keep]).astype(np.int64)

X_tr = build_tabular_np(x_train_np, tr_used)
y_tr = y_all[tr_used].astype(int)

X_val = build_tabular_np(x_train_np, va_idx)
y_val = y_all[va_idx].astype(int)

X_te  = build_tabular_np(x_test_np, te_idx)
y_te  = y_all[te_idx].astype(int)

print("Tabular shapes:", X_tr.shape, X_val.shape, X_te.shape)

results_tab = []

if RUN_XGB:
    pip_install(["xgboost"])
    from xgboost import XGBClassifier

    xgb = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="logloss",
        scale_pos_weight=max((y_tr==0).sum() / max((y_tr==1).sum(),1), 1.0),
        random_state=SEEDS[0],
        n_jobs=-1,
    )
    xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    p_val = xgb.predict_proba(X_val)[:,1]
    p_te  = xgb.predict_proba(X_te)[:,1]

    m_val = basic_metrics(p_val, y_val)
    m_te  = basic_metrics(p_te, y_te)

    thr, _, _, _ = select_threshold_max_f1(y_val, p_val)
    m_thr = apply_threshold(y_te, p_te, thr)

    out = {"model":"XGBoost","seed":SEEDS[0],"setting":SETTING,"params":np.nan,"thr_val_maxf1":thr}
    out.update({f"val_{k}": v for k,v in m_val.items()})
    out.update({f"test_{k}": v for k,v in m_te.items()})
    out.update({f"test_thr_{k}": v for k,v in m_thr.items() if k != "thr"})
    results_tab.append(out)

if RUN_LGBM:
    pip_install(["lightgbm"])
    import lightgbm as lgb

    lgbm = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="binary",
        random_state=SEEDS[0],
        n_jobs=-1,
    )
    lgbm.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
    )

    p_val = lgbm.predict_proba(X_val)[:,1]
    p_te  = lgbm.predict_proba(X_te)[:,1]

    m_val = basic_metrics(p_val, y_val)
    m_te  = basic_metrics(p_te, y_te)

    thr, _, _, _ = select_threshold_max_f1(y_val, p_val)
    m_thr = apply_threshold(y_te, p_te, thr)

    out = {"model":"LightGBM","seed":SEEDS[0],"setting":SETTING,"params":np.nan,"thr_val_maxf1":thr}
    out.update({f"val_{k}": v for k,v in m_val.items()})
    out.update({f"test_{k}": v for k,v in m_te.items()})
    out.update({f"test_thr_{k}": v for k,v in m_thr.items() if k != "thr"})
    results_tab.append(out)

if results_tab:
    display(pd.DataFrame(results_tab))


In [ ]:
# %% [11] Rodar experimentos (GNNs + MLP + opcional XGB/LGBM) e salvar CSV
import pandas as pd
import gc
import torch

def cleanup_cuda():
    """Limpa cache CUDA entre execuções para reduzir risco de OOM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results = []
purity_logs = []

# ===== Ajuste aqui (capacidade) =====
HIDDEN_DIM = 128
LAYERS = 2

def make_sage():
    enc = SAGEEncoder(IN_NODE, HIDDEN_DIM, layers=LAYERS, p=DROPOUT)
    return GNNEdgeModel(enc, HIDDEN_DIM, IN_EDGE, p=DROPOUT)

def make_gat():
    enc = GATv2Encoder(IN_NODE, HIDDEN_DIM, layers=LAYERS, heads=2, p=DROPOUT)
    return GNNEdgeModel(enc, HIDDEN_DIM, IN_EDGE, p=DROPOUT)

def make_gine():
    enc = GINEEncoder(IN_NODE, IN_EDGE, HIDDEN_DIM, layers=LAYERS, p=DROPOUT)
    return GNNEdgeModel(enc, HIDDEN_DIM, IN_EDGE, p=DROPOUT)

def make_pna():
    # PNA é pesado (aggr min/max/std + scalers + towers). Habilite só se tiver memória.
    deg_hist = compute_deg_hist(mp_ei_train, num_nodes=x_train.shape[0]).to(DEVICE)
    enc = PNAEncoder(
        IN_NODE, HIDDEN_DIM,
        deg_hist=deg_hist,
        layers=LAYERS,
        towers=PNA_TOWERS,
        aggr=PNA_AGGR,
        scalers=PNA_SCALERS,
        p=DROPOUT,
    )
    return GNNEdgeModel(enc, HIDDEN_DIM, IN_EDGE, p=DROPOUT)

MODEL_CTORS = {
    "SAGE": make_sage,
    "GATv2": make_gat,
    "GINE": make_gine,
}
if INCLUDE_PNA:
    MODEL_CTORS["PNA"] = make_pna

# ===========================
# Rodar GNNs
# ===========================
if RUN_GNNS:
    for seed in SEEDS:
        for name, ctor in MODEL_CTORS.items():
            cleanup_cuda()
            try:
                out, purity_df = train_gnn(
                    model_name=name,
                    model_ctor=ctor,
                    seed=seed,
                    monitor=MONITOR,
                    max_epochs=MAX_EPOCHS,
                    patience=PATIENCE,
                    eval_every=EVAL_EVERY,
                    lr=LR,
                    wd=WD,
                    balance_train=BALANCE_TRAIN,
                    chunk_size_eval=EVAL_CHUNK_SIZE,
                )
            except (torch.cuda.OutOfMemoryError, torch.OutOfMemoryError) as e:
                print(f"[OOM] {name} | seed={seed} — pulando. Erro: {e}")
                print("      Sugestão: reduza HIDDEN_DIM/LAYERS/PNA_TOWERS, reduza EVAL_CHUNK_SIZE, ou desabilite INCLUDE_PNA.")
                cleanup_cuda()
                continue

            results.append(out)
            purity_df["model"] = name
            purity_df["seed"]  = seed
            purity_logs.append(purity_df)
            cleanup_cuda()

# ===========================
# Rodar baseline MLP (tabular, sem message passing)
# ===========================
if RUN_MLP_BASELINE:
    for seed in SEEDS:
        cleanup_cuda()
        out = train_mlp_baseline(
            seed=seed,
            h=256,
            p=DROPOUT,
            lr=LR,
            wd=0.0,
            max_epochs=50,
            patience=5,
            eval_every=2,
            balance_train=BALANCE_TRAIN,
            chunk_size_eval=EVAL_CHUNK_SIZE,
        )
        results.append(out)
        cleanup_cuda()

# resultados de baselines tabulares (XGB/LGBM), se executados
if "results_tab" in globals() and isinstance(results_tab, list) and len(results_tab) > 0:
    results.extend(results_tab)

df_res = pd.DataFrame(results)
display(df_res.sort_values("val_AP", ascending=False).head(10))

out_csv = RESULTS_DIR / "results_v4_clean.csv"
df_res.to_csv(out_csv, index=False)
print("[OK] Salvo:", out_csv)

if purity_logs:
    df_purity = pd.concat(purity_logs, ignore_index=True)
    df_purity.to_csv(RESULTS_DIR / "purity_v4_clean.csv", index=False)
    print("[OK] Salvo:", RESULTS_DIR / "purity_v4_clean.csv")


In [ ]:
# %% [12] Relatórios: tabelas (média±dp por modelo), PR/ROC, pureza vs k e custo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_DIR / "results_v4_clean.csv")
print("Linhas:", len(df))
display(df.head(3))

metric_cols = [c for c in df.columns if c.startswith("test_") and ("thr_" not in c)]
group = df.groupby("model")[metric_cols].agg(["mean","std"])
display(group)

latex_path = RESULTS_DIR / "table_results_v4_clean.tex"
with open(latex_path, "w", encoding="utf-8") as f:
    f.write(group.to_latex(float_format=lambda x: f"{x:.4f}"))
print("[OK] LaTeX:", latex_path)

# Seleciona melhor rodagem por val_AP (sem olhar teste)
best = df.sort_values("val_AP", ascending=False).iloc[0]
best_model = best["model"]
best_seed  = int(best["seed"])
print("Melhor por val_AP:", best_model, "seed", best_seed)

PROBS_DIR = RESULTS_DIR / "probs_v4"
val_npz  = np.load(PROBS_DIR / f"{best_model}_seed{best_seed}_val.npz")
test_npz = np.load(PROBS_DIR / f"{best_model}_seed{best_seed}_test.npz")
y_val, p_val = val_npz["y"].astype(int), val_npz["p"].astype(float)
y_te,  p_te  = test_npz["y"].astype(int), test_npz["p"].astype(float)

from sklearn.metrics import precision_recall_curve, roc_curve

# PR curve
prec, rec, _ = precision_recall_curve(y_te, p_te)
plt.figure()
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"PR curve — {best_model} (test)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / f"fig_pr_{best_model}_test.png", dpi=200)
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_te, p_te)
plt.figure()
plt.plot(fpr, tpr)
plt.xlabel("FPR (FAR)")
plt.ylabel("TPR (Recall)")
plt.title(f"ROC curve — {best_model} (test)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / f"fig_roc_{best_model}_test.png", dpi=200)
plt.show()

# Tabela Recall x FAR por grid de thresholds (descritivo; NÃO usar para escolher modelo)
thr_grid = np.round(np.arange(0.30, 0.401, 0.01), 2)
rows = []
neg = (y_te == 0)
pos = (y_te == 1)
for thr in thr_grid:
    yhat = (p_te >= thr).astype(int)
    far = float(((yhat == 1) & neg).sum() / max(int(neg.sum()), 1))
    rec_thr = float(((yhat == 1) & pos).sum() / max(int(pos.sum()), 1))
    rows.append({"thr": float(thr), "FAR": far, "Recall": rec_thr})
df_thr = pd.DataFrame(rows)
display(df_thr)

df_thr.to_csv(RESULTS_DIR / f"table_recall_far_thrgrid_{best_model}_test.csv", index=False)

# Pureza vs k% (se existe)
pur_path = RESULTS_DIR / "purity_v4_clean.csv"
if pur_path.exists():
    pur = pd.read_csv(pur_path)
    pur_best = pur[(pur["model"]==best_model) & (pur["seed"]==best_seed)].sort_values("k%")
    display(pur_best)

    plt.figure()
    plt.plot(pur_best["k%"], pur_best["purity_macro_induced"], marker="o", label="macro (induced)")
    plt.plot(pur_best["k%"], pur_best["purity_weighted_induced"], marker="o", label="weighted (induced)")
    plt.plot(pur_best["k%"], pur_best["purity_macro_selected"], marker="x", label="macro (selected)")
    plt.plot(pur_best["k%"], pur_best["purity_weighted_selected"], marker="x", label="weighted (selected)")
    plt.xlabel("Orçamento k (%)")
    plt.ylabel("Pureza / contaminação")
    plt.title(f"Pureza de grupos — {best_model} (test)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"fig_group_purity_{best_model}_test.png", dpi=200)
    plt.show()
else:
    print("[INFO] Arquivo de pureza não encontrado:", pur_path)

# Custo real
cost_cols = ["train_seconds","rss_gb_start","rss_gb_end","gpu_peak_gb"]
cost = df.groupby("model")[cost_cols].agg(["mean","std"])
display(cost)

print("[OK] Figuras em:", PLOTS_DIR)


## (Opcional) Sweep de profundidade / épocas / seeds

Use esta seção apenas se você realmente precisar investigar profundidade. Em geral, **2–4 camadas** é uma faixa mais defensável para GNNs padrão.

Se você insistir em testar profundidades altas (ex.: 20–100), mantenha:

- **residual**
- **LayerNorm**
- **dropout**
- monitoramento por **AP/Recall@k** na validação (early stopping)


In [ ]:
# %% [13] (Opcional) Sweep de profundidade (SAGE residual) — NÃO rodar por padrão
RUN_DEPTH_SWEEP = False  # mude para True para rodar

if RUN_DEPTH_SWEEP:
    depths = [2, 4, 8, 16, 32]  # evite 100 como primeiro teste
    hidden = 128

    sweep_rows = []
    for d in depths:
        def make_sage_depth():
            enc = SAGEEncoder(IN_NODE, hidden, layers=d, p=DROPOUT)
            return GNNEdgeModel(enc, hidden, IN_EDGE, p=DROPOUT)

        for seed in SEEDS:
            out, _ = train_gnn(
                model_name=f"SAGE_depth{d}",
                model_ctor=make_sage_depth,
                seed=seed,
                monitor=MONITOR,
                max_epochs=MAX_EPOCHS,
                patience=PATIENCE,
                eval_every=EVAL_EVERY,
                lr=LR,
                wd=WD,
                balance_train=BALANCE_TRAIN,
            )
            out["depth"] = d
            sweep_rows.append(out)

    df_sweep = pd.DataFrame(sweep_rows)
    display(df_sweep.sort_values("val_AP", ascending=False).head(10))
    df_sweep.to_csv(RESULTS_DIR / "sweep_depth_v4_clean.csv", index=False)
    print("[OK] Salvo:", RESULTS_DIR / "sweep_depth_v4_clean.csv")
else:
    print("[SKIP] RUN_DEPTH_SWEEP=False")


# (Novo) Experimento adicional: executar o pipeline final (GraphSAGE) no dataset AML1M

Esta seção **não altera** o que foi executado anteriormente (AML100k). Ela apenas adiciona um *segundo bloco* ao final do notebook que:

1. Carrega (ou reconstrói) o artefato `edge_data_v4_clean.pt` no diretório `AML1M`;
2. Roda **apenas** o modelo escolhido (GraphSAGE) no dataset `AML1M`;
3. Exporta estatísticas descritivas do dataset (AML1M), além de tabelas/figuras e um relatório de custo (tempo/memória) para enriquecer o Capítulo 5;
4. (Opcional) Exporta um exemplo de *caso* (componente conectada) do top-*k* para ilustração qualitativa.

**Observação:** se o `AML1M` for grande a ponto de estourar a GPU em *full-batch*, você pode:
- reduzir `AML1M_HIDDEN_DIM`;
- definir `AML1M_DEVICE = torch.device("cpu")` (mais lento, mas evita OOM na GPU);
- ou executar esta seção em uma GPU maior (A100 40/80GB, H100, etc.).


In [ ]:
# %% [AML1M-0] Configuração do experimento AML1M (GraphSAGE only)

RUN_AML1M = True  # defina False para pular esta seção

# Caminho do dataset AML1M no Google Drive (conforme você indicou)
AML1M_PROJECT_DIR = BASE / "DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M"
AML1M_DATA_DIR    = AML1M_PROJECT_DIR
AML1M_ARTIF_DIR   = AML1M_PROJECT_DIR / "artifacts"

# Saídas separadas (não misturam com AML100k)
AML1M_TAG         = "aml1m_graphsage_only"
AML1M_RESULTS_DIR = AML1M_PROJECT_DIR / f"results_{AML1M_TAG}"
AML1M_PLOTS_DIR   = AML1M_PROJECT_DIR / f"plots_{AML1M_TAG}"
AML1M_OUTPUT_DIR  = AML1M_PROJECT_DIR / f"report_outputs_{AML1M_TAG}"

AML1M_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
AML1M_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
(AML1M_OUTPUT_DIR / "tables").mkdir(parents=True, exist_ok=True)
(AML1M_OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
AML1M_ARTIF_DIR.mkdir(parents=True, exist_ok=True)

AML1M_CACHE_PATH  = AML1M_ARTIF_DIR / "edge_data_v4_clean.pt"

# Seeds e hiperparâmetros (por padrão: 1 seed para reduzir custo)
AML1M_SEEDS       = [44]        # ajuste se quiser média±dp (ex.: [42, 43, 44])
AML1M_MAX_EPOCHS  = MAX_EPOCHS
AML1M_PATIENCE   = PATIENCE
AML1M_EVAL_EVERY = EVAL_EVERY
AML1M_LR         = LR
AML1M_WD         = WD
AML1M_BAL_TRAIN  = BALANCE_TRAIN

# Capacidade do modelo (pode reduzir caso dê OOM)
AML1M_HIDDEN_DIM  = HIDDEN_DIM
AML1M_LAYERS      = LAYERS
AML1M_DROPOUT     = DROPOUT

# Para datasets muito grandes, às vezes é necessário reduzir o chunk de inferência
AML1M_CHUNK_SIZE_EVAL = CHUNK_SIZE_EVAL

# Se quiser forçar reconstrução do cache do AML1M:
AML1M_FORCE_REBUILD = False

# Se quiser depurar com subconjunto (NÃO usar para resultados finais):
AML1M_DEBUG_MAX_EDGES = None   # ex.: 2_000_000

# Device do AML1M (pode setar CPU para evitar OOM em full-batch)
AML1M_DEVICE = DEVICE  # ex.: torch.device("cpu")

print("[AML1M] PROJECT_DIR:", AML1M_PROJECT_DIR)
print("[AML1M] CACHE_PATH :", AML1M_CACHE_PATH)
print("[AML1M] RESULTS_DIR:", AML1M_RESULTS_DIR)
print("[AML1M] PLOTS_DIR  :", AML1M_PLOTS_DIR)
print("[AML1M] OUTPUT_DIR :", AML1M_OUTPUT_DIR)

# Backup do contexto atual (para opcionalmente restaurar no final)
_context_backup = {
    "PROJECT_DIR": PROJECT_DIR,
    "DATA_DIR": DATA_DIR,
    "ARTIF_DIR": ARTIF_DIR,
    "CACHE_PATH": CACHE_PATH,
    "RESULTS_DIR": RESULTS_DIR,
    "PLOTS_DIR": PLOTS_DIR,
    "OUTPUT_DIR": OUTPUT_DIR if "OUTPUT_DIR" in globals() else None,
    "FORCE_REBUILD": FORCE_REBUILD,
    "DEBUG_MAX_EDGES": DEBUG_MAX_EDGES,
    "DEVICE": DEVICE,
    # variáveis de dataset/cache (restauração completa é opcional)
    "x_train": x_train,
    "x_test": x_test,
    "mp_ei_train": mp_ei_train,
    "mp_ei_test": mp_ei_test,
    "mp_ea_train": mp_ea_train,
    "mp_ea_test": mp_ea_test,
    "ei_all_cpu": ei_all_cpu,
    "ea_all_cpu": ea_all_cpu,
    "y_all_cpu": y_all_cpu,
    "tr_idx": tr_idx,
    "va_idx": va_idx,
    "te_idx": te_idx,
}


In [ ]:
# %% [AML1M-1] Carregar/Construir cache do AML1M + estatísticas descritivas (sem misturar com AML100k)

if not RUN_AML1M:
    print("[AML1M] RUN_AML1M=False — pulando.")
else:
    import time, math, os, gc
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import torch
    from pathlib import Path

    # ----------------------------
    # Utilitários de custo
    # ----------------------------
    aml1m_cost_rows = []

    def _reset_gpu_peak():
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def log_cost(stage, t0, extra=None):
        row = {
            "stage": stage,
            "seconds": float(time.perf_counter() - t0),
            "rss_gb": float(mem_rss_gb()),
            "gpu_peak_gb": float(gpu_peak_gb()),
        }
        if extra:
            row.update(extra)
        aml1m_cost_rows.append(row)
        print(f"[AML1M|COST] {stage}: {row['seconds']:.1f}s | rss={row['rss_gb']:.2f}GB | gpu_peak={row['gpu_peak_gb']:.2f}GB")

    def cleanup():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ----------------------------
    # Troca de contexto (apenas a partir daqui)
    # ----------------------------
    PROJECT_DIR  = AML1M_PROJECT_DIR
    DATA_DIR     = AML1M_DATA_DIR
    ARTIF_DIR    = AML1M_ARTIF_DIR
    CACHE_PATH   = AML1M_CACHE_PATH
    RESULTS_DIR  = AML1M_RESULTS_DIR
    PLOTS_DIR    = AML1M_PLOTS_DIR
    OUTPUT_DIR   = AML1M_OUTPUT_DIR
    FORCE_REBUILD = AML1M_FORCE_REBUILD
    DEBUG_MAX_EDGES = AML1M_DEBUG_MAX_EDGES

    # IMPORTANTE: se mudar DEVICE, faça antes do build do cache
    DEVICE = AML1M_DEVICE

    print("[AML1M] DEVICE:", DEVICE)
    cleanup()

    # ----------------------------
    # 1) Carregar arquivo de transações
    # ----------------------------
    t0 = time.perf_counter()

    trx_path = find_first_recursive(
        DATA_DIR,
        candidates=[
            "transactions.csv", "transaction.csv", "tx.csv",
            "hi-large_trans.csv","hi-medium_trans.csv","hi-small_trans.csv",
            "li-large_trans.csv","li-medium_trans.csv","li-small_trans.csv",
            "transactions.parquet", "transaction.parquet", "tx.parquet",
            "edges.csv", "edge.csv"
        ],
    )
    if trx_path is None:
        raise FileNotFoundError(f"[AML1M] Não encontrei arquivo de transações em: {DATA_DIR}")

    trx_path = Path(trx_path)
    ext = trx_path.suffix.lower()

    if ext == ".parquet":
        # Parquet: carrega tudo (se estiver grande demais, converta para CSV ou ajuste leitura com pyarrow)
        transactions = pd.read_parquet(trx_path)
        transactions.columns = [c.strip().lower() for c in transactions.columns]
        # Inferência de colunas (usa pick_col)
        SRC_COL = pick_col(transactions, ["source", "src", "orig", "from", "sender", "account_from", "from_id", "payer"], "SRC")
        DST_COL = pick_col(transactions, ["target", "dst", "dest", "to", "receiver", "account_to", "to_id", "payee"], "DST")
        AMT_COL = pick_col(transactions, ["amount", "amt", "value", "transaction_amount", "tx_amount"], "AMOUNT")
        TIME_COL= pick_col(transactions, ["time", "timestamp", "ts", "step", "date", "datetime"], "TIME")
        TYPE_COL= pick_col(transactions, ["type", "transaction_type", "tx_type", "category"], "TYPE")
        Y_COL   = pick_col(transactions, ["label", "is_illicit", "illicit", "y", "fraud", "suspicious"], "LABEL")
    else:
        # CSV: lê amostra para inferir colunas sem carregar tudo primeiro
        sample = pd.read_csv(trx_path, nrows=2000)
        sample.columns = [c.strip() for c in sample.columns]
        norm_names = [c.strip().lower() for c in sample.columns]
        norm_to_orig = dict(zip(norm_names, sample.columns))

        sample_norm = sample.copy()
        sample_norm.columns = norm_names

        SRC_COL = pick_col(sample_norm, ["source", "src", "orig", "from", "sender", "account_from", "from_id", "payer"], "SRC")
        DST_COL = pick_col(sample_norm, ["target", "dst", "dest", "to", "receiver", "account_to", "to_id", "payee"], "DST")
        AMT_COL = pick_col(sample_norm, ["amount", "amt", "value", "transaction_amount", "tx_amount"], "AMOUNT")
        TIME_COL= pick_col(sample_norm, ["time", "timestamp", "ts", "step", "date", "datetime"], "TIME")
        TYPE_COL= pick_col(sample_norm, ["type", "transaction_type", "tx_type", "category"], "TYPE")
        Y_COL   = pick_col(sample_norm, ["label", "is_illicit", "illicit", "y", "fraud", "suspicious"], "LABEL")

        usecols_orig = [norm_to_orig[c] for c in [SRC_COL, DST_COL, AMT_COL, TIME_COL, TYPE_COL, Y_COL]]
        transactions = pd.read_csv(trx_path, usecols=usecols_orig)
        transactions.columns = [c.strip().lower() for c in transactions.columns]

    log_cost("load_transactions_file", t0, {"path": str(trx_path), "rows": int(len(transactions))})

    # ----------------------------
    # 2) Construção do grafo (edge_index, edge_attr, y, ts)
    # ----------------------------
    t0 = time.perf_counter()

    df = transactions[[SRC_COL, DST_COL, AMT_COL, TIME_COL, TYPE_COL, Y_COL]].copy()

    df[SRC_COL] = df[SRC_COL].astype(str)
    df[DST_COL] = df[DST_COL].astype(str)
    df[Y_COL]   = df[Y_COL].astype(int)

    # timestamp -> int64
    if not np.issubdtype(df[TIME_COL].dtype, np.number):
        df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
        if df[TIME_COL].isna().any():
            raise ValueError("[AML1M] Falha ao converter TIME_COL para datetime. Verifique o formato.")
        df[TIME_COL] = df[TIME_COL].astype("int64")  # ns
    ts = df[TIME_COL].to_numpy().astype(np.int64)

    # amount -> log1p
    amt = df[AMT_COL].to_numpy().astype(np.float64)
    log_amt = np.log1p(np.maximum(amt, 0.0)).astype(np.float32)

    # type one-hot
    tx_type = df[TYPE_COL].astype(str)
    type_ohe = pd.get_dummies(tx_type, prefix="type", dtype=np.int8)
    ea_df_raw = pd.concat([pd.Series(log_amt, name="_log_amt"), type_ohe], axis=1)

    # Mapear contas -> ids (factorize é eficiente)
    E0 = len(df)
    codes, uniques = pd.factorize(pd.concat([df[SRC_COL], df[DST_COL]], axis=0), sort=False)
    src_id = codes[:E0].astype(np.int64)
    dst_id = codes[E0:].astype(np.int64)
    N = int(len(uniques))

    edge_index_np = np.stack([src_id, dst_id], axis=0).astype(np.int64)
    y_all = df[Y_COL].to_numpy().astype(np.int64)

    # remove self-loops
    not_self = edge_index_np[0] != edge_index_np[1]
    if not np.all(not_self):
        edge_index_np = edge_index_np[:, not_self]
        ea_df_raw = ea_df_raw.loc[not_self].reset_index(drop=True)
        y_all = y_all[not_self]
        ts = ts[not_self]

    # debug opcional
    E = int(edge_index_np.shape[1])
    if DEBUG_MAX_EDGES is not None and E > int(DEBUG_MAX_EDGES):
        keep = np.random.RandomState(0).choice(np.arange(E), size=int(DEBUG_MAX_EDGES), replace=False)
        keep.sort()
        edge_index_np = edge_index_np[:, keep]
        ea_df_raw = ea_df_raw.iloc[keep].reset_index(drop=True)
        y_all = y_all[keep]
        ts = ts[keep]
        E = int(edge_index_np.shape[1])
        print(f"[AML1M] DEBUG_MAX_EDGES aplicado: E={E}")

    log_cost("build_graph_arrays", t0, {"N": int(N), "E": int(E), "n_pos": int(y_all.sum()), "edge_feat_dim": int(ea_df_raw.shape[1])})

    # ----------------------------
    # 3) Split temporal (mesma lógica do notebook)
    # ----------------------------
    t0 = time.perf_counter()
    tr_mask, va_mask, te_mask = temporal_split(ts, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC)

    tr_idx = np.where(tr_mask)[0]
    va_idx = np.where(va_mask)[0]
    te_idx = np.where(te_mask)[0]

    y_tr = y_all[tr_idx]
    y_va = y_all[va_idx]
    y_te = y_all[te_idx]

    log_cost("temporal_split", t0, {
        "E_tr": int(len(tr_idx)), "E_va": int(len(va_idx)), "E_te": int(len(te_idx)),
        "pos_tr": int(y_tr.sum()), "pos_va": int(y_va.sum()), "pos_te": int(y_te.sum()),
        "prev_te": float(y_te.mean()) if len(y_te) else np.nan,
    })

    # ----------------------------
    # 4) Cache (scaler train-only + node feats sem leakage + mp graphs)
    # ----------------------------
    t0 = time.perf_counter()
    _reset_gpu_peak()

    if CACHE_PATH.exists() and (not FORCE_REBUILD):
        print("[AML1M|CACHE] Carregando:", CACHE_PATH)
        cache = torch_load_cache(CACHE_PATH, map_location="cpu")
        if cache.get("SETTING") != SETTING or cache.get("TRAIN_FRAC") != TRAIN_FRAC or cache.get("VAL_FRAC") != VAL_FRAC:
            print("[AML1M|CACHE] Config mudou (SETTING/split). Reconstruindo...")
            cache = build_all_and_cache()
            torch.save(cache, CACHE_PATH)
    else:
        print("[AML1M|CACHE] Construindo do zero...")
        cache = build_all_and_cache()
        torch.save(cache, CACHE_PATH)

    log_cost("build_or_load_cache", t0, {"cache_path": str(CACHE_PATH), "cache_size_gb": float(CACHE_PATH.stat().st_size/(1024**3))})

    # Extrai tensores para o pipeline AML1M
    x_train = cache["x_train"].to(DEVICE)
    x_test  = cache["x_test"].to(DEVICE)

    mp_ei_train = cache["mp_ei_train"].to(DEVICE)
    mp_ei_test  = cache["mp_ei_test"].to(DEVICE)

    mp_ea_train = cache["mp_ea_train"].to(DEVICE)
    mp_ea_test  = cache["mp_ea_test"].to(DEVICE)

    ei_all_cpu = cache["ei_all_cpu"]  # CPU
    ea_all_cpu = cache["ea_all_cpu"]  # CPU
    y_all_cpu  = cache["y_all_cpu"]   # CPU

    # ----------------------------
    # 5) Estatísticas descritivas básicas (AML1M)
    # ----------------------------
    t0 = time.perf_counter()

    stats = {
        "dataset": "AML1M",
        "SETTING": SETTING,
        "N_nodes": int(N),
        "E_edges": int(E),
        "train_frac": float(TRAIN_FRAC),
        "val_frac": float(VAL_FRAC),
        "E_tr": int(len(tr_idx)),
        "E_va": int(len(va_idx)),
        "E_te": int(len(te_idx)),
        "pos_tr": int(y_tr.sum()),
        "pos_va": int(y_va.sum()),
        "pos_te": int(y_te.sum()),
        "prev_tr": float(y_tr.mean()) if len(y_tr) else np.nan,
        "prev_va": float(y_va.mean()) if len(y_va) else np.nan,
        "prev_te": float(y_te.mean()) if len(y_te) else np.nan,
        "edge_feat_dim": int(ea_df_raw.shape[1]),
        "node_feat_dim": int(x_train.shape[1]),
        "ts_min": int(ts.min()) if len(ts) else None,
        "ts_max": int(ts.max()) if len(ts) else None,
        "n_types": int(type_ohe.shape[1]),
    }

    df_stats = pd.DataFrame([stats])
    display(df_stats)

    (OUTPUT_DIR/"tables"/"AML1M_dataset_stats.csv").write_text(df_stats.to_csv(index=False), encoding="utf-8")
    (OUTPUT_DIR/"tables"/"AML1M_dataset_stats.tex").write_text(df_stats.to_latex(index=False, escape=False), encoding="utf-8")

    # Sumário de amount (em log)
    amt_s = pd.Series(log_amt)
    amt_desc = amt_s.describe(percentiles=[0.5,0.9,0.99]).to_frame("log_amt")
    display(amt_desc)
    amt_desc.to_csv(OUTPUT_DIR/"tables"/"AML1M_amount_log_desc.csv")
    (OUTPUT_DIR/"tables"/"AML1M_amount_log_desc.tex").write_text(amt_desc.to_latex(escape=False), encoding="utf-8")

    log_cost("descriptive_stats", t0)

    # Salva custo parcial (até aqui)
    df_cost = pd.DataFrame(aml1m_cost_rows)
    df_cost.to_csv(OUTPUT_DIR/"tables"/"AML1M_cost_log_partial.csv", index=False)
    display(df_cost)


In [ ]:
# %% [AML1M-2] Treinar e avaliar apenas GraphSAGE no AML1M (salvando métricas, probs e custo)

if not RUN_AML1M:
    print("[AML1M] RUN_AML1M=False — pulando.")
else:
    import time
    import pandas as pd

    # Garante diretórios padrão do pipeline (mesmo padrão do AML100k, mas isolado em outra pasta)
    PROBS_DIR = RESULTS_DIR / "probs_v4"
    PROBS_DIR.mkdir(parents=True, exist_ok=True)

    # Wrapper para medir tempo das inferências dentro do train_gnn
    _predict_proba_edges_orig = predict_proba_edges
    _timing_rows = []

    def predict_proba_edges_timed(*args, **kwargs):
        t0 = time.perf_counter()
        out = _predict_proba_edges_orig(*args, **kwargs)
        dt = time.perf_counter() - t0
        try:
            idx_edges = args[4]
            n_edges = int(len(idx_edges))
        except Exception:
            n_edges = None
        _timing_rows.append({"stage": "predict_proba_edges", "seconds": float(dt), "n_edges": n_edges})
        return out

    predict_proba_edges = predict_proba_edges_timed

    # Constrói o modelo (dinâmico para tolerar possíveis mudanças de dimensão entre AML100k e AML1M)
    def make_sage_aml1m():
        enc = SAGEEncoder(x_train.shape[1], AML1M_HIDDEN_DIM, layers=AML1M_LAYERS, p=AML1M_DROPOUT)
        return GNNEdgeModel(enc, AML1M_HIDDEN_DIM, ea_all_cpu.shape[1], p=AML1M_DROPOUT)

    results = []
    purity_logs = []

    for seed in AML1M_SEEDS:
        cleanup_cuda()  # do notebook original
        t0 = time.perf_counter()

        out, purity_df = train_gnn(
            model_name="GraphSAGE",
            model_ctor=make_sage_aml1m,
            seed=int(seed),
            max_epochs=AML1M_MAX_EPOCHS,
            patience=AML1M_PATIENCE,
            eval_every=AML1M_EVAL_EVERY,
            chunk_size_eval=AML1M_CHUNK_SIZE_EVAL,
            lr=AML1M_LR,
            wd=AML1M_WD,
            balance_train=AML1M_BAL_TRAIN,
            monitor=MONITOR,
        )

        out["dataset"] = "AML1M"
        out["wall_total_seconds"] = float(time.perf_counter() - t0)

        results.append(out)
        purity_df["model"] = "GraphSAGE"
        purity_df["seed"] = int(seed)
        purity_df["dataset"] = "AML1M"
        purity_logs.append(purity_df)

        print("[AML1M] Concluído seed", seed)

    # Restaura função original
    predict_proba_edges = _predict_proba_edges_orig

    df_res = pd.DataFrame(results)
    df_res.to_csv(RESULTS_DIR / "results_v4_clean.csv", index=False)
    display(df_res)

    df_pur = pd.concat(purity_logs, ignore_index=True)
    df_pur.to_csv(RESULTS_DIR / "purity_v4_clean.csv", index=False)
    display(df_pur.head())

    # Logs de timing de inferência (val/test dentro do train_gnn)
    df_timing = pd.DataFrame(_timing_rows)
    display(df_timing)

    df_timing.to_csv(OUTPUT_DIR/"tables"/"AML1M_inference_timing_calls.csv", index=False)

    # Atualiza custo consolidado
    df_cost = pd.DataFrame(aml1m_cost_rows)
    df_cost.to_csv(OUTPUT_DIR/"tables"/"AML1M_cost_log_until_training.csv", index=False)

    print("[AML1M] Salvos:", RESULTS_DIR/"results_v4_clean.csv", "e", RESULTS_DIR/"purity_v4_clean.csv")


In [ ]:
# %% [AML1M-3] Relatórios AML1M: tabelas/figuras (curvas, top-k, grupos, custo) + exemplo de caso

if not RUN_AML1M:
    print("[AML1M] RUN_AML1M=False — pulando.")
else:
    import time, math
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

    # ----------------------------
    # 1) Carrega resultados e seleciona melhor seed (por val_AP)
    # ----------------------------
    df = pd.read_csv(RESULTS_DIR / "results_v4_clean.csv")
    display(df)

    best = df.sort_values("val_AP", ascending=False).iloc[0]
    best_model = best["model"]
    best_seed  = int(best["seed"])
    print("[AML1M] Melhor por val_AP:", best_model, "seed", best_seed)

    PROBS_DIR = RESULTS_DIR / "probs_v4"
    val_npz  = np.load(PROBS_DIR / f"{best_model}_seed{best_seed}_val.npz")
    test_npz = np.load(PROBS_DIR / f"{best_model}_seed{best_seed}_test.npz")

    y_val, p_val = val_npz["y"].astype(int), val_npz["p"].astype(float)
    y_te,  p_te  = test_npz["y"].astype(int), test_npz["p"].astype(float)

    # ----------------------------
    # 2) Curvas globais (ROC/PR)
    # ----------------------------
    auc = float(roc_auc_score(y_te, p_te))
    ap  = float(average_precision_score(y_te, p_te))
    base_prev = float(y_te.mean())
    print(f"[AML1M] Test AUC={auc:.4f} | AP={ap:.4f} | prevalência={base_prev:.6f} | AP/base={ap/max(base_prev,1e-12):.1f}x")

    # ROC
    fpr, tpr, _ = roc_curve(y_te, p_te)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"ROC — {best_model} (AML1M test)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"AML1M_fig_ROC_{best_model}_seed{best_seed}.png", dpi=200)
    plt.show()

    # PR
    prec, rec, _ = precision_recall_curve(y_te, p_te)
    plt.figure()
    plt.plot(rec, prec)
    plt.axhline(base_prev, linestyle="--")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"PR — {best_model} (AML1M test)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"AML1M_fig_PR_{best_model}_seed{best_seed}.png", dpi=200)
    plt.show()

    # ----------------------------
    # 3) Métricas top-k (k em {1,2,5,10}%)
    # ----------------------------
    ks_pct = [1,2,5,10]
    ks = [k/100.0 for k in ks_pct]
    tm = topk_metrics(p_te, y_te, ks=ks)

    rows = []
    E_te = len(y_te)
    pos_te = int(y_te.sum())
    for k in ks:
        k_pct = int(k*100)
        Ek = int(math.ceil(E_te*k))
        rows.append({
            "k%": f"{k_pct}%",
            "|E_k|": Ek,
            "Recall@k": float(tm[f"Recall@{k_pct}%"]),
            "Precision@k": float(tm[f"Precision@{k_pct}%"]),
            "Lift@k": float(tm[f"Lift@{k_pct}%"]),
            "Inspecoes_por_ilicito": float(1.0/max(tm[f"Precision@{k_pct}%"], 1e-12)),
        })

    df_topk = pd.DataFrame(rows)
    display(df_topk)

    df_topk.to_csv(OUTPUT_DIR/"tables"/"AML1M_topk_transacao.csv", index=False)
    (OUTPUT_DIR/"tables"/"AML1M_topk_transacao.tex").write_text(df_topk.to_latex(index=False, escape=False), encoding="utf-8")

    # curvas k (recall/prec/lift)
    k_grid = np.array(ks_pct, dtype=float)
    rec_k = np.array([tm[f"Recall@{k}%"] for k in ks_pct], dtype=float)
    prec_k = np.array([tm[f"Precision@{k}%"] for k in ks_pct], dtype=float)
    lift_k = np.array([tm[f"Lift@{k}%"] for k in ks_pct], dtype=float)

    plt.figure()
    plt.plot(k_grid, rec_k, marker="o")
    plt.xlabel("Orçamento k (%)")
    plt.ylabel("Recall@k")
    plt.title(f"Recall@k — {best_model} (AML1M test)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"AML1M_fig_Recall_k_{best_model}_seed{best_seed}.png", dpi=200)
    plt.show()

    plt.figure()
    plt.plot(k_grid, prec_k, marker="o")
    plt.xlabel("Orçamento k (%)")
    plt.ylabel("Precision@k")
    plt.title(f"Precision@k — {best_model} (AML1M test)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"AML1M_fig_Precision_k_{best_model}_seed{best_seed}.png", dpi=200)
    plt.show()

    plt.figure()
    plt.plot(k_grid, lift_k, marker="o")
    plt.xlabel("Orçamento k (%)")
    plt.ylabel("Lift@k")
    plt.title(f"Lift@k — {best_model} (AML1M test)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"AML1M_fig_Lift_k_{best_model}_seed{best_seed}.png", dpi=200)
    plt.show()

    # ----------------------------
    # 4) Fase 2 — grupos: pureza induzida + cobertura + CR@k
    # ----------------------------
    te_idx_np = te_idx.numpy() if hasattr(te_idx, "numpy") else np.array(te_idx)
    edge_index_np_all = ei_all_cpu.numpy() if hasattr(ei_all_cpu, "numpy") else np.array(ei_all_cpu)
    y_all_np = y_all_cpu.numpy() if hasattr(y_all_cpu, "numpy") else np.array(y_all_cpu)

    phase2_rows = []
    for k in ks_pct:
        t0 = time.perf_counter()
        gp = group_purity_stats(
            y_edge=y_all_np[te_idx_np],
            edge_index_2xE=edge_index_np_all[:, te_idx_np],
            scores=p_te,
            k_pct=float(k),
            min_nodes=3,
        )
        # cobertura: ilícitos induzidos / ilícitos totais no teste
        cov = float(gp["pos_in_groups_induced"] / max(int(pos_te), 1))
        rec = float(tm[f"Recall@{k}%"])
        cr  = float(rec / max(cov, 1e-12)) if not np.isnan(cov) else np.nan

        Ek = int(math.ceil((k/100.0) * len(te_idx_np)))

        phase2_rows.append({
            "k%": f"{k}%",
            "|E_k|": Ek,
            "#casos": int(gp["n_groups"]),
            "Recall@k": rec,
            "Pureza_induzida(weighted)": float(gp["purity_weighted_induced"]),
            "Cobertura(casos)": cov,
            "CR@k": cr,
            "|E_ind|": int(gp["edges_in_groups_induced"]),
            "|Y+_ind|": int(gp["pos_in_groups_induced"]),
            "E_ind/E_k": float(gp["edges_in_groups_induced"] / max(Ek, 1)),
            "seconds_group_stats": float(time.perf_counter() - t0),
        })

    df_phase2 = pd.DataFrame(phase2_rows)
    display(df_phase2)

    df_phase2.to_csv(OUTPUT_DIR/"tables"/"AML1M_phase2_grupos.csv", index=False)
    (OUTPUT_DIR/"tables"/"AML1M_phase2_grupos.tex").write_text(df_phase2.to_latex(index=False, escape=False), encoding="utf-8")

    # Trade-off plot (pureza induzida vs cobertura)
    plt.figure()
    plt.plot(df_phase2["Cobertura(casos)"], df_phase2["Pureza_induzida(weighted)"], marker="o")
    for _, r in df_phase2.iterrows():
        plt.annotate(r["k%"], (r["Cobertura(casos)"], r["Pureza_induzida(weighted)"]))
    plt.xlabel("Cobertura (casos induzidos)")
    plt.ylabel("Pureza induzida (ponderada)")
    plt.title(f"Trade-off pureza vs cobertura — {best_model} (AML1M)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"AML1M_fig_tradeoff_pureza_cobertura_{best_model}_seed{best_seed}.png", dpi=200)
    plt.show()

    # ----------------------------
    # 5) Custo — sumariza métricas do results_v4_clean.csv
    # ----------------------------
    cost_cols = ["train_seconds","wall_total_seconds","rss_gb_start","rss_gb_end","gpu_peak_gb","best_epoch"]
    df_cost = df[cost_cols].copy()
    display(df_cost)

    df_cost.to_csv(OUTPUT_DIR/"tables"/"AML1M_cost_summary.csv", index=False)
    (OUTPUT_DIR/"tables"/"AML1M_cost_summary.tex").write_text(df_cost.to_latex(index=False, escape=False), encoding="utf-8")

    # ----------------------------
    # 6) (Opcional) Exemplo qualitativo de caso no top-1% (salva figura simplificada)
    # ----------------------------
    EXPORT_CASE_EXAMPLE = False  # coloque True se quiser gerar a figura (pode ser custoso em grafos muito grandes)
    MAX_NODES_PLOT = 120
    MAX_EDGES_FOR_PLOT_GRAPH = 200_000  # amostra de arestas do top-k para não explodir memória
    K_EXAMPLE = 1  # top-1%

    if EXPORT_CASE_EXAMPLE:
        import networkx as nx

        rng = np.random.RandomState(0)

        order = np.argsort(-p_te)
        K = int(math.ceil((K_EXAMPLE/100.0) * len(te_idx_np)))
        sel_local = order[:K]
        sel_global_edges = te_idx_np[sel_local]

        src_full = edge_index_np_all[0, sel_global_edges]
        dst_full = edge_index_np_all[1, sel_global_edges]
        y_sel_full = y_all_np[sel_global_edges].astype(int)

        # escolhe aresta positiva (se houver)
        pos_idx = np.where(y_sel_full == 1)[0]
        i0 = int(pos_idx[0]) if len(pos_idx) else int(rng.randint(0, len(src_full)))
        u0, v0 = int(src_full[i0]), int(dst_full[i0])

        # amostra arestas do top-k (garante incluir a aresta i0)
        if len(src_full) > MAX_EDGES_FOR_PLOT_GRAPH:
            rest = np.setdiff1d(np.arange(len(src_full)), np.array([i0]), assume_unique=False)
            sample_rest = rng.choice(rest, size=MAX_EDGES_FOR_PLOT_GRAPH-1, replace=False)
            sample_idx = np.concatenate([np.array([i0]), sample_rest])
            src = src_full[sample_idx]
            dst = dst_full[sample_idx]
        else:
            src, dst = src_full, dst_full

        G = nx.Graph()
        G.add_edges_from(zip(src.tolist(), dst.tolist()))

        # BFS limitado
        nodes = set([u0, v0])
        frontier = [u0, v0]
        while frontier and len(nodes) < MAX_NODES_PLOT:
            u = frontier.pop(0)
            for w in list(G.neighbors(u)):
                if w not in nodes:
                    nodes.add(w)
                    frontier.append(w)
                    if len(nodes) >= MAX_NODES_PLOT:
                        break

        H = G.subgraph(nodes).copy()

        plt.figure(figsize=(8,6))
        pos = nx.spring_layout(H, seed=0)
        nx.draw_networkx_nodes(H, pos, node_size=80)
        nx.draw_networkx_edges(H, pos, width=1.0, alpha=0.8)
        plt.title(f"Exemplo de caso (AML1M) — top-{K_EXAMPLE}% (amostrado, BFS limitado)")
        plt.axis("off")
        out_path = PLOTS_DIR / f"AML1M_case_example_top{K_EXAMPLE}.png"
        plt.tight_layout()
        plt.savefig(out_path, dpi=200)
        plt.show()
        print("[AML1M] Figura exemplo salva em:", out_path)

    print("[AML1M] Saídas em:", RESULTS_DIR, PLOTS_DIR, OUTPUT_DIR)


In [ ]:
# %% [AML1M-4] (Opcional) Restaurar contexto AML100k (útil se você quiser continuar usando variáveis do primeiro experimento)

RESTORE_AML100K_CONTEXT = False

if RUN_AML1M and RESTORE_AML100K_CONTEXT:
    PROJECT_DIR = _context_backup["PROJECT_DIR"]
    DATA_DIR    = _context_backup["DATA_DIR"]
    ARTIF_DIR   = _context_backup["ARTIF_DIR"]
    CACHE_PATH  = _context_backup["CACHE_PATH"]
    RESULTS_DIR = _context_backup["RESULTS_DIR"]
    PLOTS_DIR   = _context_backup["PLOTS_DIR"]
    if _context_backup["OUTPUT_DIR"] is not None:
        OUTPUT_DIR = _context_backup["OUTPUT_DIR"]
    FORCE_REBUILD = _context_backup["FORCE_REBUILD"]
    DEBUG_MAX_EDGES = _context_backup["DEBUG_MAX_EDGES"]
    DEVICE = _context_backup["DEVICE"]

    x_train = _context_backup["x_train"]
    x_test  = _context_backup["x_test"]
    mp_ei_train = _context_backup["mp_ei_train"]
    mp_ei_test  = _context_backup["mp_ei_test"]
    mp_ea_train = _context_backup["mp_ea_train"]
    mp_ea_test  = _context_backup["mp_ea_test"]
    ei_all_cpu  = _context_backup["ei_all_cpu"]
    ea_all_cpu  = _context_backup["ea_all_cpu"]
    y_all_cpu   = _context_backup["y_all_cpu"]
    tr_idx = _context_backup["tr_idx"]
    va_idx = _context_backup["va_idx"]
    te_idx = _context_backup["te_idx"]

    print("[OK] Contexto AML100k restaurado.")
else:
    print("[INFO] RESTORE_AML100K_CONTEXT=False (ou RUN_AML1M=False).")
